# Phase 2: structural equation models and Monte Carlo

Run all cells in order with the same 71-item CSV schema. Test whether the proposed chain of associations is present; do not infer causation.


## 1. Connect Google Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive connected.")
except ImportError:
    print("Outside Google Colab: skip the Drive mount.")

## 2. Set input and output paths


In [ ]:
import os

INPUT_CSV_PATH = os.environ.get(
    'PHASE2_INPUT_CSV',
    '/content/drive/MyDrive/SOICT-2026/data/soict_2026_survey_responses.csv',
)
OUTPUT_BASE_DIR = os.environ.get(
    'PHASE2_OUTPUT_DIR',
    '/content/drive/MyDrive/SOICT-2026/results/phase_02',
)

CFI_TARGET, TLI_TARGET, RMSEA_LIMIT, SRMR_LIMIT = 0.95, 0.95, 0.08, 0.08
DELTA_CFI_LIMIT, DELTA_RMSEA_LIMIT = -0.010, 0.015
P_LIMIT, VCOV_EIGEN_TOLERANCE = 0.05, -1e-8
REFERENCE_TOLERANCE = 0.001
SENSITIVITY_BETA_DIFFERENCE = 0.10
RANDOM_SEED = 20260908
MC_DRAWS, MC_SEED, CI_LEVEL = 100_000, 20260908, 0.95

print('Survey CSV:', INPUT_CSV_PATH)
print('Results directory:', OUTPUT_BASE_DIR)


## 3. Install dependencies


In [ ]:
import importlib.util
import subprocess
import sys
from shutil import which

required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


def run_checked(command, description):
    print(description)
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-6000:])
    if result.returncode != 0:
        raise RuntimeError(f"Failed: {description}")
    return result.stdout


if which("Rscript") is None:
    run_checked(["apt-get", "update", "-qq"], "Updating system package lists.")
    run_checked(["apt-get", "install", "-y", "-qq", "r-base"], "Installing R.")
check_r = subprocess.run(
    ["Rscript", "-e", "quit(status=ifelse(requireNamespace('lavaan', quietly=TRUE),0,1))"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
if check_r.returncode != 0:
    run_checked(
        ["Rscript", "-e", "install.packages('lavaan', repos='https://cloud.r-project.org')"],
        "Installing lavaan.",
    )
run_checked(
    ["Rscript", "-e", "library(lavaan); cat(as.character(packageVersion('lavaan')))"],
    "Checking the lavaan version.",
)
version_text = run_checked(
    ['Rscript', '-e', "library(lavaan); cat(as.character(packageVersion('lavaan')))"] ,
    'Verify the lavaan version',
).strip().splitlines()[-1]
parts = tuple(int(part) for part in version_text.replace('-', '.').split('.')[:3])
if parts < (0, 7, 2):
    run_checked(
        ['Rscript', '-e', "install.packages('lavaan', repos='https://cloud.r-project.org')"],
        'Update lavaan for Monte Carlo',
    )
    version_text = run_checked(
        ['Rscript', '-e', "library(lavaan); cat(as.character(packageVersion('lavaan')))"] ,
        'Recheck lavaan',
    ).strip().splitlines()[-1]
    parts = tuple(int(part) for part in version_text.replace('-', '.').split('.')[:3])
    if parts < (0, 7, 2):
        raise RuntimeError(f'lavaan {version_text} is too old; version >= 0.7-2 is required.')


## 4. Define the measurement structure


In [ ]:
from __future__ import annotations

import hashlib
import html
import json
import shutil
import zipfile
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid")

SCALE_ITEMS = {
    "ENP": [f"ENP{i:02d}" for i in range(1, 7)],
    "PTED": [f"PTED{i:02d}" for i in range(1, 7)],
    "MEU": [f"MEU{i:02d}" for i in range(1, 6)],
    "REH": [f"REH{i:02d}" for i in range(1, 4)],
    "ELA": [f"ELA{i:02d}" for i in range(1, 4)],
    "ORG": [f"ORG{i:02d}" for i in range(1, 4)],
    "CRT": [f"CRT{i:02d}" for i in range(1, 4)],
    "MSR": [f"MSR{i:02d}" for i in range(1, 5)],
    "TSEM": [f"TSEM{i:02d}" for i in range(1, 4)],
    "VLS": [f"VLS{i:02d}" for i in range(1, 7)],
    "DEL": [f"DEL{i:02d}" for i in range(1, 8)],
    "LPSN": [f"LPSN{i:02d}" for i in range(1, 6)],
    "SFN": [f"SFN{i:02d}" for i in range(1, 5)],
    "TECN": [f"TECN{i:02d}" for i in range(1, 7)],
    "PU": [f"PU{i:02d}" for i in range(1, 5)],
    "BI": [f"BI{i:02d}" for i in range(1, 4)],
}

TIER2_SCALES = ["REH", "ELA", "ORG", "CRT", "MSR", "TSEM", "VLS", "DEL"]
TIER3_SCALES = ["LPSN", "SFN", "TECN"]
ALL_ITEMS = [item for items in SCALE_ITEMS.values() for item in items]
NON_ENP_ITEMS = [item for item in ALL_ITEMS if item not in SCALE_ITEMS["ENP"]]

assert len(SCALE_ITEMS) == 16
assert len(ALL_ITEMS) == 71 and len(set(ALL_ITEMS)) == 71

## 5. Validate the input CSV


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def read_csv_auto(path):
    errors = []
    for encoding in ('utf-8-sig', 'utf-8', 'cp1258', 'cp1252', 'latin1'):
        for separator in (None, ',', ';', '\t'):
            try:
                options = dict(dtype=str, encoding=encoding, keep_default_na=True)
                options.update(dict(sep=separator, engine='python') if separator is None else dict(sep=separator))
                frame = pd.read_csv(path, **options)
                if set(ALL_ITEMS).issubset(set(str(col).strip() for col in frame.columns)):
                    return frame, encoding, separator if separator is not None else 'auto-detected'
            except Exception as exc:
                errors.append(f'{encoding}/{separator}: {exc}')
    raise ValueError('The CSV is unreadable or does not contain all 71 coded items. ' + ' | '.join(errors[-3:]))


input_path = Path(INPUT_CSV_PATH).expanduser().resolve()
if not input_path.is_file():
    raise FileNotFoundError(f'Survey CSV not found: {input_path}')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir = Path(OUTPUT_BASE_DIR).expanduser() / f'phase_02_{stamp}'
output_dir.mkdir(parents=True, exist_ok=False)
input_hash_before_r = sha256_file(input_path)
integrity_path = input_path

data, csv_encoding, csv_separator = read_csv_auto(input_path)
data.columns = [str(column).strip() for column in data.columns]
if data.columns.duplicated().any() or len(data) < 10:
    raise ValueError('CSV column names must be unique and the sample must be nonempty.')
missing_items = [item for item in ALL_ITEMS if item not in data.columns]
if missing_items:
    raise ValueError(f'Missing item codes: {missing_items}')
if 'RESPONDENT_ID' in data.columns and (data.RESPONDENT_ID.isna().any() or data.RESPONDENT_ID.duplicated().any()):
    raise ValueError('RESPONDENT_ID values must be present and unique.')

for item in ALL_ITEMS:
    original = data[item].copy()
    converted = pd.to_numeric(original, errors='coerce')
    if (original.notna() & converted.isna()).any():
        raise ValueError(f'{item}: contains a response that is not numeric.')
    if not converted.dropna().isin([1, 2, 3, 4, 5]).all():
        raise ValueError(f'{item}: contains a response outside the Likert range 1 to 5.')
    data[item] = converted

enp_missing = data[SCALE_ITEMS['ENP']].isna().sum(axis=1)
if not enp_missing.isin([0, 6]).all():
    raise ValueError('All six ENP items must be observed or absent according to survey routing.')
if data[NON_ENP_ITEMS].isna().any().any():
    raise ValueError('A Likert item outside ENP is missing.')
if 'S0_ENGLISH_NECESSARY' in data.columns:
    original_s0 = data['S0_ENGLISH_NECESSARY']
    s0 = pd.to_numeric(original_s0, errors='coerce')
    if (original_s0.notna() & s0.isna()).any() or not s0.isin([0, 1]).all():
        raise ValueError('S0_ENGLISH_NECESSARY must be fully observed and binary.')
    if not bool(((s0 == 0) == (enp_missing == 6)).all()):
        raise ValueError('S0 does not match the ENP survey routing.')

matrix = data[ALL_ITEMS].copy().reset_index(drop=True)
n_total = len(matrix)
n_primary = int(matrix.notna().all(axis=1).sum())
n_enp_missing = int((enp_missing == 6).sum())
if n_primary < 10:
    raise ValueError('Insufficient fully observed respondents for the primary model.')

shutil.copy2(input_path, output_dir / '00_ban_sao_du_lieu_nguon.csv')
matrix.to_csv(output_dir / '00_du_lieu_71_cau_cho_R.csv', index=False)
checks = pd.DataFrame([
    ['SHA-256 CSV', input_hash_before_r, True],
    ['Input respondents', n_total, True],
    ['Primary sample complete on 71 items', n_primary, n_primary > 0],
    ['Respondents for whom ENP was not applicable', n_enp_missing, True],
    ['All 71 Likert item columns are valid', len(ALL_ITEMS), True],
    ['Missing values outside ENP', int(data[NON_ENP_ITEMS].isna().sum().sum()), True],
], columns=['check', 'value', 'passed'])
checks.to_csv(output_dir / '01_kiem_tra_dau_vao.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([{
    'csv_file': str(input_path), 'sha256': input_hash_before_r,
    'encoding': csv_encoding, 'separator': csv_separator,
    'n_total': n_total, 'n_complete_71': n_primary, 'n_enp_not_applicable': n_enp_missing,
}]).to_csv(output_dir / '00_dau_vet_dau_vao.csv', index=False, encoding='utf-8-sig')
print(f'Input validated: {n_total} respondents; {n_primary} complete on 71 items; {n_enp_missing} routed away from ENP.')
display(checks)


## 6. Specify four structural models


In [ ]:
def measurement_syntax(scales):
    lines = []
    for scale in scales:
        lines.append(f"{scale} =~ {' + '.join(SCALE_ITEMS[scale])}")
    return "\n".join(lines)


measurement_all = measurement_syntax(list(SCALE_ITEMS)) + "\n" + \
    f"STRATEGY =~ {' + '.join(TIER2_SCALES)}\n" + \
    f"FUNCTIONAL_NEEDS =~ {' + '.join(TIER3_SCALES)}"

measurement_no_enp = measurement_syntax([s for s in SCALE_ITEMS if s != "ENP"]) + "\n" + \
    f"STRATEGY =~ {' + '.join(TIER2_SCALES)}\n" + \
    f"FUNCTIONAL_NEEDS =~ {' + '.join(TIER3_SCALES)}"

partial_structure = r'''
STRATEGY ~ a_enp*ENP + a_pted*PTED + a_meu*MEU
FUNCTIONAL_NEEDS ~ b_strategy*STRATEGY + d_enp_need*ENP + d_pted_need*PTED + d_meu_need*MEU
PU ~ c_need*FUNCTIONAL_NEEDS + d_strategy_pu*STRATEGY + d_enp_pu*ENP + d_pted_pu*PTED + d_meu_pu*MEU
BI ~ g_pu*PU + d_need_bi*FUNCTIONAL_NEEDS + d_strategy_bi*STRATEGY + d_enp_bi*ENP + d_pted_bi*PTED + d_meu_bi*MEU
ENP ~~ PTED + MEU
PTED ~~ MEU

serial_enp := a_enp*b_strategy*c_need*g_pu
serial_pted := a_pted*b_strategy*c_need*g_pu
serial_meu := a_meu*b_strategy*c_need*g_pu
serial_strategy := b_strategy*c_need*g_pu
serial_need := c_need*g_pu
ind_enp_to_need_via_strategy := a_enp*b_strategy
ind_pted_to_need_via_strategy := a_pted*b_strategy
ind_meu_to_need_via_strategy := a_meu*b_strategy
ind_enp_to_pu_via_chain := a_enp*b_strategy*c_need
ind_pted_to_pu_via_chain := a_pted*b_strategy*c_need
ind_meu_to_pu_via_chain := a_meu*b_strategy*c_need
'''

serial_structure = r'''
STRATEGY ~ a_enp*ENP + a_pted*PTED + a_meu*MEU
FUNCTIONAL_NEEDS ~ b_strategy*STRATEGY
PU ~ c_need*FUNCTIONAL_NEEDS
BI ~ g_pu*PU
ENP ~~ PTED + MEU
PTED ~~ MEU

serial_enp := a_enp*b_strategy*c_need*g_pu
serial_pted := a_pted*b_strategy*c_need*g_pu
serial_meu := a_meu*b_strategy*c_need*g_pu
serial_strategy := b_strategy*c_need*g_pu
serial_need := c_need*g_pu
'''

reverse_structure = r'''
FUNCTIONAL_NEEDS ~ r_enp*ENP + r_pted*PTED + r_meu*MEU
STRATEGY ~ r_need_strategy*FUNCTIONAL_NEEDS
PU ~ r_strategy_pu*STRATEGY
BI ~ r_pu_bi*PU
ENP ~~ PTED + MEU
PTED ~~ MEU

reverse_enp := r_enp*r_need_strategy*r_strategy_pu*r_pu_bi
reverse_pted := r_pted*r_need_strategy*r_strategy_pu*r_pu_bi
reverse_meu := r_meu*r_need_strategy*r_strategy_pu*r_pu_bi
'''

no_enp_structure = r'''
STRATEGY ~ a_pted*PTED + a_meu*MEU
FUNCTIONAL_NEEDS ~ b_strategy*STRATEGY + d_pted_need*PTED + d_meu_need*MEU
PU ~ c_need*FUNCTIONAL_NEEDS + d_strategy_pu*STRATEGY + d_pted_pu*PTED + d_meu_pu*MEU
BI ~ g_pu*PU + d_need_bi*FUNCTIONAL_NEEDS + d_strategy_bi*STRATEGY + d_pted_bi*PTED + d_meu_bi*MEU
PTED ~~ MEU

serial_pted := a_pted*b_strategy*c_need*g_pu
serial_meu := a_meu*b_strategy*c_need*g_pu
'''

model_syntax = {
    "SEM_PARTIAL_PRIMARY": measurement_all + "\n" + partial_structure,
    "SEM_SERIAL_ONLY": measurement_all + "\n" + serial_structure,
    "SEM_REVERSE_SERIAL": measurement_all + "\n" + reverse_structure,
    "SEM_PARTIAL_NO_ENP": measurement_no_enp + "\n" + no_enp_structure,
}

(output_dir / "00_dac_ta_bon_mo_hinh_lavaan.txt").write_text(
    "\n\n".join(f"### {name}\n{syntax}" for name, syntax in model_syntax.items()),
    encoding="utf-8",
)

## 7. Estimate ordered categorical SEM in lavaan


In [ ]:
def r_vector(values):
    return "c(" + ", ".join(json.dumps(v) for v in values) + ")"


r_models = "list(\n" + ",\n".join(
    f"  {json.dumps(name)}={json.dumps(syntax)}" for name, syntax in model_syntax.items()
) + "\n)"

r_cfa_syntax = json.dumps(measurement_all)

r_script = f'''
suppressPackageStartupMessages(library(lavaan))
options(warn=1)
set.seed({RANDOM_SEED})

output_dir <- {json.dumps(str(output_dir))}
dat <- read.csv(file.path(output_dir, "00_du_lieu_71_cau_cho_R.csv"), check.names=FALSE)
all_items <- {r_vector(ALL_ITEMS)}
non_enp_items <- {r_vector(NON_ENP_ITEMS)}
model_syntax <- {r_models}

fit_one <- function(label, syntax, block, ordered_items) {{
  captured <- character()
  error_message <- NA_character_
  fit <- tryCatch(
    withCallingHandlers(
      sem(
        model=syntax, data=block, ordered=ordered_items,
        estimator="WLSMV", parameterization="theta",
        std.lv=TRUE, missing="listwise", fixed.x=FALSE
      ),
      warning=function(w) {{
        captured <<- c(captured, conditionMessage(w))
        invokeRestart("muffleWarning")
      }}
    ),
    error=function(e) {{ error_message <<- conditionMessage(e); NULL }}
  )
  converged <- !is.null(fit) && isTRUE(lavInspect(fit, "converged"))
  post_ok <- if (converged) isTRUE(lavInspect(fit, "post.check")) else FALSE
  vcov_min <- NA_real_
  if (converged) {{
    vcm <- tryCatch(
      withCallingHandlers(vcov(fit), warning=function(w) {{
        captured <<- c(captured, conditionMessage(w)); invokeRestart("muffleWarning")
      }}), error=function(e) NULL
    )
    if (!is.null(vcm) && nrow(vcm) > 0) vcov_min <- min(eigen((vcm+t(vcm))/2, symmetric=TRUE, only.values=TRUE)$values)
  }}
  cat("MO_HINH:", label, "; N:", nrow(block), "; HOI_TU:", converged, "; HAU_NGHIEM:", post_ok, "; VCOV_MIN:", vcov_min, "\\n")
  if (length(captured)) cat("CANH_BAO:", label, ":", paste(unique(captured), collapse=" || "), "\\n")
  if (!is.na(error_message)) cat("LOI:", label, ":", error_message, "\\n")
  list(label=label, fit=fit, converged=converged, post_ok=post_ok, vcov_min=vcov_min,
       warnings=unique(captured), error=error_message, n=nrow(block))
}}

extract_fit <- function(result) {{
  if (is.null(result$fit) || !isTRUE(result$converged)) return(data.frame(
    mo_hinh=result$label, n=result$n, hoi_tu=FALSE, hau_nghiem_dat=FALSE,
    npar=NA_real_, chisq_scaled=NA_real_, df_scaled=NA_real_, p_scaled=NA_real_,
    cfi_scaled=NA_real_, tli_scaled=NA_real_, rmsea_scaled=NA_real_, srmr=NA_real_,
    phuong_sai_am=NA_integer_, vcov_tri_rieng_nho_nhat=result$vcov_min,
    so_canh_bao=length(result$warnings), loi=result$error, stringsAsFactors=FALSE
  ))
  fm <- fitMeasures(result$fit)
  getfm <- function(name) if (name %in% names(fm)) unname(fm[name]) else NA_real_
  pe <- parameterEstimates(result$fit)
  variances <- pe[pe$op == "~~" & pe$lhs == pe$rhs, "est"]
  data.frame(
    mo_hinh=result$label, n=result$n, hoi_tu=TRUE, hau_nghiem_dat=result$post_ok,
    npar=getfm("npar"), chisq_scaled=getfm("chisq.scaled"), df_scaled=getfm("df.scaled"),
    p_scaled=getfm("pvalue.scaled"), cfi_scaled=getfm("cfi.scaled"), tli_scaled=getfm("tli.scaled"),
    rmsea_scaled=getfm("rmsea.scaled"), srmr=getfm("srmr"),
    phuong_sai_am=sum(variances < -1e-6, na.rm=TRUE),
    vcov_tri_rieng_nho_nhat=result$vcov_min, so_canh_bao=length(result$warnings),
    loi=NA_character_, stringsAsFactors=FALSE
  )
}}

extract_parameters <- function(result) {{
  if (is.null(result$fit) || !isTRUE(result$converged)) return(data.frame())
  pe <- parameterEstimates(result$fit, standardized=TRUE, ci=TRUE)
  pe$mo_hinh <- result$label
  pe
}}

extract_r2 <- function(result) {{
  if (is.null(result$fit) || !isTRUE(result$converged)) return(data.frame())
  values <- tryCatch(lavInspect(result$fit, "r2"), error=function(e) numeric())
  values <- unlist(values)
  if (!length(values)) return(data.frame())
  data.frame(mo_hinh=result$label, bien_phu_thuoc=names(values), R2=as.numeric(values), stringsAsFactors=FALSE)
}}

primary <- dat[complete.cases(dat[, all_items]), , drop=FALSE]
if (nrow(primary) != {n_primary}) stop("Primary sample size differs from the validated input.")
if (nrow(dat) != {n_total}) stop("Sensitivity sample size differs from the validated input.")

models <- list(
  SEM_PARTIAL_PRIMARY=fit_one("SEM_PARTIAL_PRIMARY", model_syntax$SEM_PARTIAL_PRIMARY, primary, all_items),
  SEM_SERIAL_ONLY=fit_one("SEM_SERIAL_ONLY", model_syntax$SEM_SERIAL_ONLY, primary, all_items),
  SEM_REVERSE_SERIAL=fit_one("SEM_REVERSE_SERIAL", model_syntax$SEM_REVERSE_SERIAL, primary, all_items),
  SEM_PARTIAL_NO_ENP=fit_one("SEM_PARTIAL_NO_ENP", model_syntax$SEM_PARTIAL_NO_ENP, dat, non_enp_items)
)

# Refit CFA on the same CSV and sample to check measurement consistency.
cfa_reference <- fit_one("FULL_STRATEGY_NEED_SECOND_ORDER_RECHECK", {r_cfa_syntax}, primary, all_items)
write.csv(extract_fit(cfa_reference), file.path(output_dir, "02_CFA_kiem_tra_noi_bo.csv"),
          row.names=FALSE, fileEncoding="UTF-8")
if (!is.null(cfa_reference$fit) && isTRUE(cfa_reference$converged)) {{
  capture.output(summary(cfa_reference$fit, fit.measures=TRUE, standardized=TRUE),
                 file=file.path(output_dir, "00_tom_tat_CFA_noi_bo.txt"))
}} else {{
  writeLines("Internal CFA did not converge.", file.path(output_dir, "00_tom_tat_CFA_noi_bo.txt"))
}}

fit_table <- do.call(rbind, lapply(models, extract_fit))
write.csv(fit_table, file.path(output_dir, "02_do_phu_hop_bon_mo_hinh_SEM.csv"), row.names=FALSE, fileEncoding="UTF-8")

parameters <- do.call(rbind, lapply(models, extract_parameters))
if (is.null(parameters)) parameters <- data.frame()
write.csv(parameters, file.path(output_dir, "03_toan_bo_tham_so_SEM.csv"), row.names=FALSE, fileEncoding="UTF-8")

if (nrow(parameters)) {{
  paths <- parameters[parameters$op == "~", ]
  indirects <- parameters[parameters$op == ":=", ]
}} else {{ paths <- data.frame(); indirects <- data.frame() }}
write.csv(paths, file.path(output_dir, "04_he_so_duong_dan.csv"), row.names=FALSE, fileEncoding="UTF-8")
write.csv(indirects, file.path(output_dir, "05_tac_dong_gian_tiep.csv"), row.names=FALSE, fileEncoding="UTF-8")

r2_table <- do.call(rbind, lapply(models, extract_r2))
if (is.null(r2_table)) r2_table <- data.frame()
write.csv(r2_table, file.path(output_dir, "06_R2_bien_noi_sinh.csv"), row.names=FALSE, fileEncoding="UTF-8")

warning_rows <- do.call(rbind, lapply(models, function(result) {{
  messages <- result$warnings
  if (!length(messages)) messages <- "No warnings"
  data.frame(mo_hinh=result$label, canh_bao=messages, stringsAsFactors=FALSE)
}}))
write.csv(warning_rows, file.path(output_dir, "07_canh_bao_R_theo_mo_hinh.csv"), row.names=FALSE, fileEncoding="UTF-8")

lrt_error <- NA_character_
lrt <- tryCatch({{
  if (isTRUE(models$SEM_SERIAL_ONLY$converged) && isTRUE(models$SEM_PARTIAL_PRIMARY$converged)) {{
    as.data.frame(lavTestLRT(models$SEM_SERIAL_ONLY$fit, models$SEM_PARTIAL_PRIMARY$fit))
  }} else data.frame()
}}, error=function(e) {{ lrt_error <<- conditionMessage(e); data.frame() }})
if (!nrow(lrt)) lrt <- data.frame(thong_bao=ifelse(is.na(lrt_error), "Too few converged models for comparison", lrt_error))
write.csv(lrt, file.path(output_dir, "08_kiem_dinh_chenh_lech_WLSMV.csv"), row.names=TRUE, fileEncoding="UTF-8")

for (label in names(models)) {{
  result <- models[[label]]
  destination <- file.path(output_dir, paste0("00_tom_tat_", label, ".txt"))
  if (!is.null(result$fit) && isTRUE(result$converged)) {{
    capture.output(summary(result$fit, fit.measures=TRUE, standardized=TRUE, rsquare=TRUE), file=destination)
  }} else {{
    writeLines(ifelse(is.na(result$error), "The model did not converge.", result$error), con=destination)
  }}
}}

capture.output(sessionInfo(), file=file.path(output_dir, "00_thong_tin_moi_truong_R.txt"))
cat("The R portion of phase 2 completed.\\n")
'''

r_script_path = output_dir / "00_chay_notebook_2_SEM.R"
r_script_path.write_text(r_script, encoding="utf-8")
print("Generated R script:", r_script_path)

In [ ]:
fit_path = output_dir / '02_do_phu_hop_bon_mo_hinh_SEM.csv'
with open(output_dir / '00_nhat_ky_R.txt', 'w', encoding='utf-8') as log_file:
    result = subprocess.run(['Rscript', str(r_script_path)], text=True,
                            stdout=log_file, stderr=subprocess.STDOUT)
log_path = output_dir / '00_nhat_ky_R.txt'
print(log_path.read_text(encoding='utf-8', errors='replace')[-12000:])
if result.returncode != 0:
    raise RuntimeError(f'R SEM/CFA exited with code {result.returncode}; xem {log_path}')
if not fit_path.is_file():
    raise FileNotFoundError('R completed without the required SEM fit table.')
if sha256_file(input_path) != input_hash_before_r:
    raise RuntimeError('Input CSV changed while R was running.')


## 8. Evaluate model fit, links, and the CFA reference


In [ ]:
def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().casefold() in {"true", "1", "yes", "yes", "co"}


def num(value):
    return pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]


fits = pd.read_csv(fit_path)
paths = pd.read_csv(output_dir / "04_he_so_duong_dan.csv")
indirects = pd.read_csv(output_dir / "05_tac_dong_gian_tiep.csv")
r2 = pd.read_csv(output_dir / "06_R2_bien_noi_sinh.csv")
lrt = pd.read_csv(output_dir / "08_kiem_dinh_chenh_lech_WLSMV.csv")

numeric_fit = ["n", "npar", "chisq_scaled", "df_scaled", "p_scaled", "cfi_scaled", "tli_scaled", "rmsea_scaled", "srmr", "phuong_sai_am", "vcov_tri_rieng_nho_nhat"]
for column in numeric_fit:
    if column in fits:
        fits[column] = pd.to_numeric(fits[column], errors="coerce")

primary_name = "SEM_PARTIAL_PRIMARY"
serial_name = "SEM_SERIAL_ONLY"
reverse_name = "SEM_REVERSE_SERIAL"
sensitivity_name = "SEM_PARTIAL_NO_ENP"
primary = fits[fits["mo_hinh"] == primary_name].iloc[0]

comparison_rows = []
for _, row in fits.iterrows():
    dcfi = row["cfi_scaled"] - primary["cfi_scaled"] if pd.notna(row["cfi_scaled"]) and pd.notna(primary["cfi_scaled"]) else np.nan
    drmsea = row["rmsea_scaled"] - primary["rmsea_scaled"] if pd.notna(row["rmsea_scaled"]) and pd.notna(primary["rmsea_scaled"]) else np.nan
    absolute_ok = bool(
        as_bool(row["hoi_tu"]) and as_bool(row["hau_nghiem_dat"])
        and row["phuong_sai_am"] == 0
        and row["vcov_tri_rieng_nho_nhat"] >= VCOV_EIGEN_TOLERANCE
        and row["cfi_scaled"] >= CFI_TARGET and row["tli_scaled"] >= TLI_TARGET
        and row["rmsea_scaled"] <= RMSEA_LIMIT and row["srmr"] <= SRMR_LIMIT
    ) if pd.notna(row["cfi_scaled"]) else False
    comparison_rows.append({
        **row.to_dict(), "delta_CFI_so_voi_mo_hinh_chinh": dcfi,
        "delta_RMSEA_so_voi_mo_hinh_chinh": drmsea,
        "do_phu_hop_tuyet_doi_dat": absolute_ok,
        "suy_giam_tuong_doi_chap_nhan": bool(pd.notna(dcfi) and pd.notna(drmsea) and dcfi >= DELTA_CFI_LIMIT and drmsea <= DELTA_RMSEA_LIMIT),
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(output_dir / "09_so_sanh_bon_mo_hinh_SEM.csv", index=False, encoding="utf-8-sig")

cfa_reference = pd.read_csv(output_dir / '02_CFA_kiem_tra_noi_bo.csv')
if len(cfa_reference) != 1 or int(cfa_reference.iloc[0]['n']) != n_primary:
    raise ValueError('Internal CFA must produce exactly one result on the primary sample.')
old = cfa_reference.iloc[0]
reference_rows = []
for metric in ['cfi_scaled', 'tli_scaled', 'rmsea_scaled', 'srmr']:
    old_value, new_value = num(old[metric]), num(primary[metric])
    difference = new_value - old_value if pd.notna(old_value) and pd.notna(new_value) else np.nan
    reference_rows.append({
        'chi_so': metric, 'CFA_noi_bo': old_value, 'SEM_trung_gian_mot_phan': new_value,
        'chenh_lech': difference,
        'tai_lap_dat': bool(pd.notna(difference) and abs(difference) <= REFERENCE_TOLERANCE),
    })
reference_check = pd.DataFrame(reference_rows)
reference_check.to_csv(output_dir / '10_doi_chieu_CFA_noi_bo.csv', index=False, encoding='utf-8-sig')

display(comparison[["mo_hinh", "n", "hoi_tu", "cfi_scaled", "tli_scaled", "rmsea_scaled", "srmr", "delta_CFI_so_voi_mo_hinh_chinh", "delta_RMSEA_so_voi_mo_hinh_chinh", "do_phu_hop_tuyet_doi_dat"]])
display(reference_check)

In [ ]:
def select_parameter(frame, model, lhs, op, rhs=None):
    if frame.empty or not {"mo_hinh", "lhs", "op"}.issubset(frame.columns):
        return None
    rows = frame[(frame["mo_hinh"] == model) & (frame["lhs"] == lhs) & (frame["op"] == op)]
    if rhs is not None:
        rows = rows[rows["rhs"] == rhs]
    return rows.iloc[0] if len(rows) == 1 else None


def significant(row):
    if row is None:
        return False
    p = num(row.get("pvalue", np.nan))
    low = num(row.get("ci.lower", np.nan))
    high = num(row.get("ci.upper", np.nan))
    return bool(pd.notna(p) and p < P_LIMIT and pd.notna(low) and pd.notna(high) and (low > 0 or high < 0))


link_specs = [
    ("ENP → STRATEGY", "STRATEGY", "ENP"),
    ("PTED → STRATEGY", "STRATEGY", "PTED"),
    ("MEU → STRATEGY", "STRATEGY", "MEU"),
    ("STRATEGY → FUNCTIONAL_NEEDS", "FUNCTIONAL_NEEDS", "STRATEGY"),
    ("FUNCTIONAL_NEEDS → PU", "PU", "FUNCTIONAL_NEEDS"),
    ("PU → BI", "BI", "PU"),
]
link_rows = []
for label, outcome, predictor in link_specs:
    row = select_parameter(paths, primary_name, outcome, "~", predictor)
    link_rows.append({
        "mat_xich": label,
        "he_so_chua_chuan_hoa": num(row["est"]) if row is not None else np.nan,
        "beta_chuan_hoa": num(row["std.all"]) if row is not None else np.nan,
        "sai_so_chuan": num(row["se"]) if row is not None else np.nan,
        "p_value": num(row["pvalue"]) if row is not None else np.nan,
        "CI_95_duoi": num(row["ci.lower"]) if row is not None else np.nan,
        "CI_95_tren": num(row["ci.upper"]) if row is not None else np.nan,
        "co_y_nghia": significant(row),
    })
links = pd.DataFrame(link_rows)
links.to_csv(output_dir / "11_kiem_dinh_tung_mat_xich.csv", index=False, encoding="utf-8-sig")

indirect_specs = [("ENP", "serial_enp"), ("PTED", "serial_pted"), ("MEU", "serial_meu")]
indirect_rows = []
for variable, label in indirect_specs:
    row = select_parameter(indirects, primary_name, label, ":=")
    indirect_rows.append({
        "bien_tang_I": variable,
        "tac_dong": f"{variable} → STRATEGY → FUNCTIONAL_NEEDS → PU → BI",
        "uoc_luong_tich_duong_dan": num(row["est"]) if row is not None else np.nan,
        "sai_so_chuan_delta": num(row["se"]) if row is not None else np.nan,
        "p_value": num(row["pvalue"]) if row is not None else np.nan,
        "CI_95_duoi": num(row["ci.lower"]) if row is not None else np.nan,
        "CI_95_tren": num(row["ci.upper"]) if row is not None else np.nan,
        "khoang_tin_cay_khong_chua_0": significant(row),
    })
indirect_test = pd.DataFrame(indirect_rows)
indirect_test.to_csv(output_dir / "12_tac_dong_gian_tiep_day_du_theo_tang_I.csv", index=False, encoding="utf-8-sig")

core_names = ["STRATEGY → FUNCTIONAL_NEEDS", "FUNCTIONAL_NEEDS → PU", "PU → BI"]
core_ok = bool(links[links["mat_xich"].isin(core_names)]["co_y_nghia"].all()) if len(links) else False
chain_rows = []
for variable, label in indirect_specs:
    first_ok = bool(links.loc[links["mat_xich"] == f"{variable} → STRATEGY", "co_y_nghia"].iloc[0])
    indirect_ok = bool(indirect_test.loc[indirect_test["bien_tang_I"] == variable, "khoang_tin_cay_khong_chua_0"].iloc[0])
    chain_rows.append({
        "bien_tang_I": variable,
        "duong_vao_STRATEGY_dat": first_ok,
        "ba_mat_xich_phia_sau_dat": core_ok,
        "tac_dong_gian_tiep_day_du_dat": indirect_ok,
        "ket_luan_chuoi": "Statistical evidence supports the full association chain" if first_ok and core_ok and indirect_ok else "Insufficient evidence for the full association chain",
    })
chain_conclusions = pd.DataFrame(chain_rows)
chain_conclusions.to_csv(output_dir / "13_ket_luan_chuoi_theo_bien_tang_I.csv", index=False, encoding="utf-8-sig")

display(links)
display(indirect_test)
display(chain_conclusions)

In [ ]:
def extract_lrt_p(frame):
    for column in frame.columns:
        if "Pr(" in str(column) or str(column).casefold() in {"p", "pvalue", "p_value"}:
            values = pd.to_numeric(frame[column], errors="coerce").dropna()
            if len(values):
                return float(values.iloc[-1])
    return np.nan


serial_row = comparison[comparison["mo_hinh"] == serial_name].iloc[0]
reverse_row = comparison[comparison["mo_hinh"] == reverse_name].iloc[0]
lrt_p = extract_lrt_p(lrt)

serial_absolute = bool(serial_row["do_phu_hop_tuyet_doi_dat"])
serial_relative = bool(serial_row["suy_giam_tuong_doi_chap_nhan"])
lrt_no_loss = bool(pd.notna(lrt_p) and lrt_p >= P_LIMIT)

if serial_absolute and serial_relative and lrt_no_loss:
    serial_status = "The pure chain fits adequately relative to the partial mediation model"
elif serial_absolute:
    serial_status = "The pure chain has acceptable absolute fit, but partial mediation fits better"
else:
    serial_status = "The pure chain does not fit the data adequately"

serial_decision = pd.DataFrame([{
    "do_phu_hop_tuyet_doi_dat": serial_absolute,
    "suy_giam_tuong_doi_chap_nhan": serial_relative,
    "p_kiem_dinh_chenh_lech_WLSMV": lrt_p,
    "khong_giam_phu_hop_co_y_nghia": lrt_no_loss,
    "ket_luan": serial_status,
}])
serial_decision.to_csv(output_dir / "14_danh_gia_mo_hinh_chuoi_thuan_tuy.csv", index=False, encoding="utf-8-sig")

forward_cfi, reverse_cfi = num(serial_row["cfi_scaled"]), num(reverse_row["cfi_scaled"])
forward_rmsea, reverse_rmsea = num(serial_row["rmsea_scaled"]), num(reverse_row["rmsea_scaled"])
if pd.notna(forward_cfi) and pd.notna(reverse_cfi):
    if forward_cfi - reverse_cfi >= 0.01 or reverse_rmsea - forward_rmsea >= 0.015:
        direction_status = "The proposed order fits better than the reversed order"
    elif reverse_cfi - forward_cfi >= 0.01 or forward_rmsea - reverse_rmsea >= 0.015:
        direction_status = "The reversed order fits better; reexamine the proposed order"
    else:
        direction_status = "The two orders fit similarly; cross sectional data do not clearly separate them"
else:
    direction_status = "Insufficient converged models to compare ordering"

direction = pd.DataFrame([{
    "CFI_chuoi_thuan": forward_cfi, "CFI_chuoi_dao": reverse_cfi,
    "RMSEA_chuoi_thuan": forward_rmsea, "RMSEA_chuoi_dao": reverse_rmsea,
    "ket_luan": direction_status,
    "gioi_han": "Fit comparisons do not establish temporal causality from cross sectional data.",
}])
direction.to_csv(output_dir / "15_kiem_tra_mo_hinh_dao_chieu.csv", index=False, encoding="utf-8-sig")

common_paths = [
    ("STRATEGY", "PTED"), ("STRATEGY", "MEU"),
    ("FUNCTIONAL_NEEDS", "STRATEGY"), ("PU", "FUNCTIONAL_NEEDS"), ("BI", "PU"),
]
sensitivity_rows = []
for outcome, predictor in common_paths:
    p1 = select_parameter(paths, primary_name, outcome, "~", predictor)
    p2 = select_parameter(paths, sensitivity_name, outcome, "~", predictor)
    b1 = num(p1["std.all"]) if p1 is not None else np.nan
    b2 = num(p2["std.all"]) if p2 is not None else np.nan
    difference = b2 - b1 if pd.notna(b1) and pd.notna(b2) else np.nan
    sensitivity_rows.append({
        "duong_dan": f"{predictor} → {outcome}", "beta_mau_du_ENP": b1,
        "beta_mau_khong_ENP": b2, "chenh_lech": difference,
        "on_dinh_theo_nguong_mo_ta": bool(pd.notna(difference) and abs(difference) <= SENSITIVITY_BETA_DIFFERENCE),
    })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(output_dir / "16_do_nhay_khong_ENP.csv", index=False, encoding="utf-8-sig")

display(serial_decision)
display(direction)
display(sensitivity)

In [ ]:
primary_valid = bool(comparison.loc[comparison["mo_hinh"] == primary_name, "do_phu_hop_tuyet_doi_dat"].iloc[0])
reference_reproduced = bool(reference_check["tai_lap_dat"].all()) if len(reference_check) else False
all_sensitivity_stable = bool(sensitivity["on_dinh_theo_nguong_mo_ta"].all()) if len(sensitivity) else False

summary_rows = [
    {"noi_dung": "Measurement limitations", "ket_luan": "Interpret with phase 1 results", "y_nghia": "Internal CFA does not replace the scale separation assessment."},
    {"noi_dung": "Partial mediation SEM", "ket_luan": "Valid fit" if primary_valid else "Invalid fit", "y_nghia": "Technical conditions for interpreting structural paths."},
    {"noi_dung": "Internal CFA reference on the same CSV", "ket_luan": "Passed" if reference_reproduced else "Not passed", "y_nghia": "Compare CFA and SEM refitted on the same input data."},
    {"noi_dung": "Three downstream links", "ket_luan": "Statistically supported" if core_ok else "Not jointly supported", "y_nghia": "STRATEGY → FUNCTIONAL_NEEDS → PU → BI."},
    {"noi_dung": "Pure chain model", "ket_luan": serial_status, "y_nghia": "Tests the need for additional direct paths."},
    {"noi_dung": "Order comparison", "ket_luan": direction_status, "y_nghia": "A sensitivity analysis on cross sectional data."},
    {"noi_dung": f"Sensitivity using {n_total} respondents without ENP", "ket_luan": "Common paths are stable" if all_sensitivity_stable else "At least one path changed substantially", "y_nghia": f"Evaluate the effect of {n_enp_missing} respondents for whom ENP is not applicable."},
]
for _, row in chain_conclusions.iterrows():
    summary_rows.append({"noi_dung": f"Chain starting at {row['bien_tang_I']}", "ket_luan": row["ket_luan_chuoi"], "y_nghia": f"{row['bien_tang_I']} → STRATEGY → FUNCTIONAL_NEEDS → PU → BI"})

final_summary = pd.DataFrame(summary_rows)
final_summary.to_csv(output_dir / "17_ket_luan_tong_hop_SEM.csv", index=False, encoding="utf-8-sig")

guide = f'''# How to read phase 2 results

## Reading order

1. `17_ket_luan_tong_hop_SEM.csv`: overall conclusions.
2. `11_kiem_dinh_tung_mat_xich.csv`: beta and p values for six primary paths.
3. `12_tac_dong_gian_tiep_day_du_theo_tang_I.csv`: full indirect effects for ENP, PTED, and MEU.
4. `13_ket_luan_chuoi_theo_bien_tang_I.csv`: which tier I association chains meet the stated criteria.
5. `14_danh_gia_mo_hinh_chuoi_thuan_tuy.csv`: whether the pure chain needs additional direct paths.
6. `15_kiem_tra_mo_hinh_dao_chieu.csv`: whether the proposed and reversed orders are distinguishable.
7. `16_do_nhay_khong_ENP.csv`: sensitivity when using {n_total} respondents.

## Interpretation rules

- Report software p values displayed as zero using p < 0.001.
- The partial mediation structural model is nearly saturated; good fit alone does not establish the proposed chain.
- Assess each tier I chain only when its incoming path, three downstream links, and full indirect effect meet the criteria.
- Cross sectional observations support structural associations and indirect estimates, not a temporal causal sequence.

## Current technical findings

- Primary model valid: {primary_valid}
- Three downstream links jointly supported: {core_ok}
- {serial_status}
- {direction_status}
'''
(output_dir / "18_huong_dan_doc_ket_qua.md").write_text(guide, encoding="utf-8")

# Monochrome HTML report.
report = f'''<!doctype html><html lang="en"><head><meta charset="utf-8"><title>SEM report</title>
<style>body{{font-family:Arial,sans-serif;color:#000;background:#fff;max-width:1200px;margin:32px auto;line-height:1.55}}table{{border-collapse:collapse;width:100%;margin:12px 0 24px}}th,td{{border:1px solid #000;padding:8px;text-align:left;vertical-align:top}}th{{font-weight:bold}}h1,h2{{color:#000}}</style></head><body>
<h1>Phase 2: structural equation model assessment</h1>
<h2>Objective</h2><p>Test the proposed association chains from ENP, PTED, and MEU through STRATEGY, FUNCTIONAL_NEEDS, and PU to BI.</p>
<h2>Overall results</h2>{final_summary.to_html(index=False, border=1, escape=True)}
<h2>Structural links</h2>{links.to_html(index=False, border=1, escape=True)}
<h2>Full indirect effects</h2>{indirect_test.to_html(index=False, border=1, escape=True)}
<h2>Limitations</h2><p>Cross sectional observations do not establish temporal causality.</p>
</body></html>'''
(output_dir / "19_bao_cao_tong_hop_SEM.html").write_text(report, encoding="utf-8")

# Model fit figures.
plot_fit = comparison[pd.to_numeric(comparison["cfi_scaled"], errors="coerce").notna()].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
if len(plot_fit):
    sns.barplot(data=plot_fit, y="mo_hinh", x="cfi_scaled", color="white", edgecolor="black", ax=axes[0])
    axes[0].axvline(CFI_TARGET, color="black", linestyle="--", linewidth=1)
    axes[0].set_title("CFI across SEM models"); axes[0].set_xlabel("CFI"); axes[0].set_ylabel("")
    sns.barplot(data=plot_fit, y="mo_hinh", x="rmsea_scaled", color="white", edgecolor="black", ax=axes[1])
    axes[1].axvline(RMSEA_LIMIT, color="black", linestyle="--", linewidth=1)
    axes[1].set_title("RMSEA across SEM models"); axes[1].set_xlabel("RMSEA"); axes[1].set_ylabel("")
else:
    for axis in axes:
        axis.text(0.5, 0.5, "No SEM result available", ha="center", va="center"); axis.set_axis_off()
plt.tight_layout(); plt.savefig(output_dir / "21_bieu_do_do_phu_hop_SEM.png", dpi=180, bbox_inches="tight", facecolor="white"); plt.close()

# Primary structural coefficient plot.
fig, ax = plt.subplots(figsize=(11, 6))
plot_links = links[pd.to_numeric(links["beta_chuan_hoa"], errors="coerce").notna()].copy()
if len(plot_links):
    sns.barplot(data=plot_links, y="mat_xich", x="beta_chuan_hoa", color="white", edgecolor="black", ax=ax)
    ax.axvline(0, color="black", linewidth=1); ax.set_title("Standardized coefficients of primary links"); ax.set_xlabel("Standardized beta"); ax.set_ylabel("")
else:
    ax.text(0.5, 0.5, "No structural coefficients available", ha="center", va="center"); ax.set_axis_off()
plt.tight_layout(); plt.savefig(output_dir / "22_bieu_do_he_so_duong_dan.png", dpi=180, bbox_inches="tight", facecolor="white"); plt.close()

# Explained variance for core endogenous variables.
fig, ax = plt.subplots(figsize=(11, 6))
plot_r2 = r2[(r2.get("mo_hinh", pd.Series(dtype=str)) == primary_name) & r2.get("bien_phu_thuoc", pd.Series(dtype=str)).isin(["STRATEGY", "FUNCTIONAL_NEEDS", "PU", "BI"])].copy() if len(r2) else pd.DataFrame()
if len(plot_r2):
    sns.barplot(data=plot_r2, y="bien_phu_thuoc", x="R2", color="white", edgecolor="black", ax=ax)
    ax.set_xlim(0, 1); ax.set_title("Explained variance in the primary model"); ax.set_xlabel("R²"); ax.set_ylabel("")
else:
    ax.text(0.5, 0.5, "R-squared is unavailable for the four endogenous variables", ha="center", va="center"); ax.set_axis_off()
plt.tight_layout(); plt.savefig(output_dir / "23_bieu_do_R2.png", dpi=180, bbox_inches="tight", facecolor="white"); plt.close()

required_outputs = [
    "00_dau_vet_dau_vao.csv", "00_ban_sao_du_lieu_nguon.csv", "00_du_lieu_71_cau_cho_R.csv", "00_dac_ta_bon_mo_hinh_lavaan.txt",
    "00_chay_notebook_2_SEM.R", "00_nhat_ky_R.txt", "00_thong_tin_moi_truong_R.txt",
    "01_kiem_tra_dau_vao.csv", "02_CFA_kiem_tra_noi_bo.csv", "00_tom_tat_CFA_noi_bo.txt", "02_do_phu_hop_bon_mo_hinh_SEM.csv",
    "03_toan_bo_tham_so_SEM.csv", "04_he_so_duong_dan.csv", "05_tac_dong_gian_tiep.csv",
    "06_R2_bien_noi_sinh.csv", "07_canh_bao_R_theo_mo_hinh.csv", "08_kiem_dinh_chenh_lech_WLSMV.csv",
    "09_so_sanh_bon_mo_hinh_SEM.csv", "10_doi_chieu_CFA_noi_bo.csv",
    "11_kiem_dinh_tung_mat_xich.csv", "12_tac_dong_gian_tiep_day_du_theo_tang_I.csv",
    "13_ket_luan_chuoi_theo_bien_tang_I.csv", "14_danh_gia_mo_hinh_chuoi_thuan_tuy.csv",
    "15_kiem_tra_mo_hinh_dao_chieu.csv", "16_do_nhay_khong_ENP.csv",
    "17_ket_luan_tong_hop_SEM.csv", "18_huong_dan_doc_ket_qua.md", "19_bao_cao_tong_hop_SEM.html",
    "21_bieu_do_do_phu_hop_SEM.png", "22_bieu_do_he_so_duong_dan.png", "23_bieu_do_R2.png",
]

verification = []
for name in required_outputs:
    path = output_dir / name
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Output is missing or empty: {name}")
    if path.suffix.lower() == ".csv":
        pd.read_csv(path)
    verification.append({"file": name, "status": "Passed", "size_bytes": path.stat().st_size})
pd.DataFrame(verification).to_csv(output_dir / "24_kiem_tra_doc_lai_dau_ra.csv", index=False, encoding="utf-8-sig")

manifest_rows = []
for path in sorted(output_dir.iterdir()):
    if path.is_file() and path.name != "25_manifest_sha256.csv":
        manifest_rows.append({"file": path.name, "sha256": sha256_file(path), "size_bytes": path.stat().st_size})
pd.DataFrame(manifest_rows).to_csv(output_dir / "25_manifest_sha256.csv", index=False, encoding="utf-8-sig")

if sha256_file(integrity_path) != input_hash_before_r:
    raise RuntimeError("Input data changed before the notebook completed.")

print("Phase 2 SEM completed.")
print("Results directory:", output_dir)
print("Primary model valid:", primary_valid)
print("Three downstream links jointly supported:", core_ok)
print(serial_status)
print(direction_status)


## 9. Refit and check indirect effects using Monte Carlo

Compare Delta, lavaan Monte Carlo, and independent multivariate Monte Carlo intervals.


In [ ]:
# Preserve computed results even when CFI or TLI fails a threshold.
# Retain SEM tables and logs when a fitted model is invalid.
if not as_bool(primary['hoi_tu']) or not as_bool(primary['hau_nghiem_dat']):
    raise RuntimeError('The primary SEM did not converge or failed diagnostic checks; inspect model 02 and the R log.')
if not np.isfinite(num(primary['vcov_tri_rieng_nho_nhat'])) or num(primary['vcov_tri_rieng_nho_nhat']) < VCOV_EIGEN_TOLERANCE:
    raise RuntimeError('The SEM parameter covariance matrix is invalid for Monte Carlo.')
version_text = run_checked(
    ['Rscript', '-e', "library(lavaan); cat(as.character(packageVersion('lavaan')))"],
    'Check lavaan version for Monte Carlo',
).strip().splitlines()[-1]
version_parts = tuple(int(x) for x in version_text.replace('-', '.').split('.')[:3])
if version_parts < (0, 7, 2):
    raise RuntimeError(f'lavaan {version_text} does not support this Monte Carlo procedure; version >= 0.7-2 is required.')

main_output_dir = output_dir
manifest_main = pd.read_csv(main_output_dir / '25_manifest_sha256.csv')
for _, row in manifest_main.iterrows():
    path = main_output_dir / row['file']
    if not path.is_file() or sha256_file(path) != str(row['sha256']):
        raise ValueError(f'SEM output changed: {path.name}')
reference_fit = pd.read_csv(main_output_dir / '02_do_phu_hop_bon_mo_hinh_SEM.csv')
reference_paths = pd.read_csv(main_output_dir / '04_he_so_duong_dan.csv')
reference_delta = pd.read_csv(main_output_dir / '12_tac_dong_gian_tiep_day_du_theo_tang_I.csv')
if set(reference_delta['bien_tang_I']) != {'ENP', 'PTED', 'MEU'}:
    raise ValueError('SEM is missing an indirect effect for ENP, PTED, or MEU.')
output_dir = main_output_dir / 'Monte_Carlo'
output_dir.mkdir(exist_ok=False)
shutil.copy2(main_output_dir / '00_du_lieu_71_cau_cho_R.csv', output_dir / '00_du_lieu_71_cau_cho_R.csv')
model_primary = model_syntax['SEM_PARTIAL_PRIMARY']
mc_input_audit = pd.DataFrame([
    ['SEM reference files', len(manifest_main), True],
    ['Source respondents', n_total, True],
    ['Complete 71-item sample', n_primary, True],
    ['ENP not applicable', n_enp_missing, True],
    ['Likert item count', len(ALL_ITEMS), len(ALL_ITEMS) == 71],
], columns=['content', 'value', 'passed'])
mc_input_audit.to_csv(output_dir / '01_kiem_tra_dau_vao.csv', index=False, encoding='utf-8-sig')


In [ ]:
r_script = r'''
suppressPackageStartupMessages(library(lavaan))
options(warn=1)

args <- commandArgs(trailingOnly=TRUE)
output_dir <- args[1]
mc_draws <- as.integer(args[2])
mc_seed <- as.integer(args[3])
ci_level <- as.numeric(args[4])
model_file <- args[5]

dat <- read.csv(file.path(output_dir, "00_du_lieu_71_cau_cho_R.csv"), check.names=FALSE)
all_items <- names(dat)
primary <- dat[complete.cases(dat[, all_items]), , drop=FALSE]
if (nrow(dat) != __N_TOTAL__L) stop("Input sample size differs from the validated count.")
if (nrow(primary) != __N_PRIMARY__L) stop("Primary SEM sample size differs from the validated count.")
if (length(all_items) != 71L) stop("Exactly 71 ordered variables are required.")

model_primary <- paste(readLines(model_file, warn=FALSE, encoding="UTF-8"), collapse="\n")
captured <- character()
fit <- withCallingHandlers(
  sem(
    model=model_primary, data=primary, ordered=all_items,
    estimator="WLSMV", parameterization="theta", std.lv=TRUE,
    missing="listwise", fixed.x=FALSE,
    se.def="monte.carlo", monte.carlo=list(R=mc_draws, seed=mc_seed)
  ),
  warning=function(w) {
    captured <<- c(captured, conditionMessage(w))
    invokeRestart("muffleWarning")
  }
)

if (!isTRUE(lavInspect(fit, "converged"))) stop("The Monte Carlo refit did not converge.")
if (!isTRUE(lavInspect(fit, "post.check"))) stop("The model failed the post estimation checks.")

vcm <- vcov(fit)
vcov_min <- min(eigen((vcm + t(vcm))/2, symmetric=TRUE, only.values=TRUE)$values)
if (!is.finite(vcov_min) || vcov_min < -1e-8) stop("The parameter covariance matrix is not positive semidefinite.")

fm <- fitMeasures(fit)
getfm <- function(name) if (name %in% names(fm)) unname(fm[name]) else NA_real_
fit_out <- data.frame(
  mo_hinh="SEM_PARTIAL_PRIMARY_MC",
  n=nrow(primary), hoi_tu=TRUE, hau_nghiem_dat=TRUE,
  cfi_scaled=getfm("cfi.scaled"), tli_scaled=getfm("tli.scaled"),
  rmsea_scaled=getfm("rmsea.scaled"), srmr=getfm("srmr"),
  vcov_tri_rieng_nho_nhat=vcov_min,
  so_canh_bao=length(unique(captured)),
  stringsAsFactors=FALSE
)
write.csv(fit_out, file.path(output_dir, "02_fit_xac_nhan.csv"), row.names=FALSE, fileEncoding="UTF-8")

pe_mc <- parameterEstimates(fit, standardized=TRUE, ci=TRUE, level=ci_level)
paths <- pe_mc[pe_mc$op == "~", ]
ind_mc <- pe_mc[pe_mc$op == ":=" & pe_mc$lhs %in% c("serial_enp", "serial_pted", "serial_meu"), ]
write.csv(paths, file.path(output_dir, "03_he_so_duong_dan_chay_lai.csv"), row.names=FALSE, fileEncoding="UTF-8")
write.csv(ind_mc, file.path(output_dir, "04_tac_dong_gian_tiep_MC_lavaan.csv"), row.names=FALSE, fileEncoding="UTF-8")

# Recompute Delta estimates in the same session for a direct check.
fit_delta <- sem(
  model=model_primary, data=primary, ordered=all_items,
  estimator="WLSMV", parameterization="theta", std.lv=TRUE,
  missing="listwise", fixed.x=FALSE, se.def="delta"
)
pe_delta <- parameterEstimates(fit_delta, standardized=TRUE, ci=TRUE, level=ci_level)
ind_delta <- pe_delta[pe_delta$op == ":=" & pe_delta$lhs %in% c("serial_enp", "serial_pted", "serial_meu"), ]
write.csv(ind_delta, file.path(output_dir, "05_tac_dong_gian_tiep_Delta_chay_lai.csv"), row.names=FALSE, fileEncoding="UTF-8")

# Draw the independent Monte Carlo sample from MVN(theta_hat, vcov_hat).
labels_needed <- c("a_enp", "a_pted", "a_meu", "b_strategy", "c_need", "g_pu")
pt <- parTable(fit)
label_rows <- lapply(labels_needed, function(lbl) pt[pt$label == lbl & pt$free > 0, , drop=FALSE])
if (any(vapply(label_rows, nrow, integer(1)) != 1L)) stop("Cannot uniquely map the six required coefficient labels.")
free_idx <- vapply(label_rows, function(x) x$free[1], integer(1))
theta <- coef(fit)[free_idx]
names(theta) <- labels_needed
sigma <- vcm[free_idx, free_idx, drop=FALSE]
sigma <- (sigma + t(sigma))/2

eig <- eigen(sigma, symmetric=TRUE)
eig_original_min <- min(eig$values)
eig_clamped <- pmax(eig$values, 0)
transform <- eig$vectors %*% diag(sqrt(eig_clamped), nrow=length(eig_clamped))
set.seed(mc_seed + 1L)
z <- matrix(rnorm(mc_draws * length(theta)), nrow=mc_draws, ncol=length(theta))
draws <- sweep(z %*% t(transform), 2, theta, "+")
colnames(draws) <- labels_needed

products <- data.frame(
  ENP=draws[, "a_enp"] * draws[, "b_strategy"] * draws[, "c_need"] * draws[, "g_pu"],
  PTED=draws[, "a_pted"] * draws[, "b_strategy"] * draws[, "c_need"] * draws[, "g_pu"],
  MEU=draws[, "a_meu"] * draws[, "b_strategy"] * draws[, "c_need"] * draws[, "g_pu"]
)

summarize_mc <- function(values, variable) {
  alpha <- 1 - ci_level
  finite <- values[is.finite(values)]
  p_neg <- (sum(finite <= 0) + 1) / (length(finite) + 1)
  p_pos <- (sum(finite >= 0) + 1) / (length(finite) + 1)
  data.frame(
    bien_tang_I=variable,
    uoc_luong_trung_binh=mean(finite),
    trung_vi=median(finite),
    sai_so_chuan_MC=sd(finite),
    CI_MC_duoi=unname(quantile(finite, alpha/2)),
    CI_MC_tren=unname(quantile(finite, 1-alpha/2)),
    p_MC_hai_phia=min(1, 2*min(p_neg, p_pos)),
    ty_le_duong=mean(finite > 0),
    so_luot_hop_le=length(finite),
    stringsAsFactors=FALSE
  )
}

manual_mc <- do.call(rbind, lapply(names(products), function(name) summarize_mc(products[[name]], name)))
write.csv(manual_mc, file.path(output_dir, "06_tac_dong_gian_tiep_MC_doc_lap.csv"), row.names=FALSE, fileEncoding="UTF-8")

# Save a thinned distribution sample for plots without a large output file.
keep <- unique(round(seq(1, mc_draws, length.out=min(15000L, mc_draws))))
plot_draws <- products[keep, , drop=FALSE]
plot_draws$lan_lay_mau <- keep
write.csv(plot_draws, file.path(output_dir, "07_mau_phan_phoi_MC_de_ve.csv"), row.names=FALSE, fileEncoding="UTF-8")

diagnostics <- data.frame(
  noi_dung=c("lavaan version", "lavaan Monte Carlo draws", "independent Monte Carlo draws", "lavaan random seed", "independent simulation seed", "confidence level", "smallest full-model vcov eigenvalue", "smallest unadjusted six-coefficient vcov eigenvalue", "number of distinct warnings"),
  gia_tri=c(as.character(packageVersion("lavaan")), mc_draws, mc_draws, mc_seed, mc_seed+1L, ci_level, vcov_min, eig_original_min, length(unique(captured))),
  stringsAsFactors=FALSE
)
write.csv(diagnostics, file.path(output_dir, "08_chan_doan_Monte_Carlo.csv"), row.names=FALSE, fileEncoding="UTF-8")
writeLines(if (length(captured)) unique(captured) else "No warnings R.", file.path(output_dir, "00_canh_bao_R.txt"))
capture.output(summary(fit, fit.measures=TRUE, standardized=TRUE, rsquare=TRUE), file=file.path(output_dir, "00_tom_tat_SEM_MC.txt"))
capture.output(sessionInfo(), file=file.path(output_dir, "00_thong_tin_moi_truong_R.txt"))
cat("SEM and both Monte Carlo checks completed.\n")
'''

r_script = r_script.replace('__N_TOTAL__', str(n_total)).replace('__N_PRIMARY__', str(n_primary))

model_path = output_dir / "00_mo_hinh_SEM_chinh_lavaan.txt"
model_path.write_text(model_primary, encoding="utf-8")
r_path = output_dir / "00_chay_notebook_2B_MC.R"
r_path.write_text(r_script, encoding="utf-8")

with open(output_dir / '00_nhat_ky_R.txt', 'w', encoding='utf-8') as mc_log:
    mc_result = subprocess.run(
        ['Rscript', str(r_path), str(output_dir), str(MC_DRAWS), str(MC_SEED), str(CI_LEVEL), str(model_path)],
        text=True, stdout=mc_log, stderr=subprocess.STDOUT,
    )
if mc_result.returncode:
    raise RuntimeError('Monte Carlo R process failed; inspect Monte_Carlo/00_nhat_ky_R.txt.')

In [ ]:
fit_new = pd.read_csv(output_dir / "02_fit_xac_nhan.csv")
paths_new = pd.read_csv(output_dir / "03_he_so_duong_dan_chay_lai.csv")
mc_lavaan = pd.read_csv(output_dir / "04_tac_dong_gian_tiep_MC_lavaan.csv")
delta_rerun = pd.read_csv(output_dir / "05_tac_dong_gian_tiep_Delta_chay_lai.csv")
mc_manual = pd.read_csv(output_dir / "06_tac_dong_gian_tiep_MC_doc_lap.csv")

primary_ref = reference_fit.loc[reference_fit["mo_hinh"] == "SEM_PARTIAL_PRIMARY"].copy()
if len(primary_ref) != 1:
    raise ValueError("The primary SEM model could not be identified uniquely in phase 2.")

metric_map = {
    "cfi_scaled": "CFI",
    "tli_scaled": "TLI",
    "rmsea_scaled": "RMSEA",
    "srmr": "SRMR",
}
fit_comparison_rows = []
for column, label in metric_map.items():
    old = float(primary_ref.iloc[0][column])
    new = float(fit_new.iloc[0][column])
    fit_comparison_rows.append([label, old, new, new-old, abs(new-old) <= REFERENCE_TOLERANCE])
fit_comparison = pd.DataFrame(fit_comparison_rows, columns=["metric", "Notebook_2", "Notebook_2B", "difference", "passed_dung_sai"])
if not fit_comparison["passed_dung_sai"].all():
    raise ValueError("Model fit changed beyond the phase 2 comparison tolerance.")
fit_comparison.to_csv(output_dir / "09_doi_chieu_do_phu_hop.csv", index=False, encoding="utf-8-sig")

path_ref_primary = reference_paths.loc[reference_paths["mo_hinh"] == "SEM_PARTIAL_PRIMARY"].copy()
path_ref_primary["key"] = path_ref_primary["lhs"].astype(str) + " ~ " + path_ref_primary["rhs"].astype(str)
paths_new["key"] = paths_new["lhs"].astype(str) + " ~ " + paths_new["rhs"].astype(str)
path_comparison = path_ref_primary[["key", "est"]].merge(paths_new[["key", "est"]], on="key", suffixes=("_Notebook_2", "_Notebook_2B"))
path_comparison["difference"] = path_comparison["est_Notebook_2B"] - path_comparison["est_Notebook_2"]
path_comparison["passed_dung_sai"] = path_comparison["difference"].abs() <= REFERENCE_TOLERANCE
if len(path_comparison) != len(path_ref_primary) or not path_comparison["passed_dung_sai"].all():
    raise ValueError("Refitted structural coefficients do not match the phase 2 estimates.")
path_comparison.to_csv(output_dir / "10_doi_chieu_he_so_duong_dan.csv", index=False, encoding="utf-8-sig")

name_map = {"serial_enp": "ENP", "serial_pted": "PTED", "serial_meu": "MEU"}
for frame in (mc_lavaan, delta_rerun):
    frame["bien_tang_I"] = frame["lhs"].map(name_map)

comparison = reference_delta[["bien_tang_I", "uoc_luong_tich_duong_dan", "CI_95_duoi", "CI_95_tren"]].rename(columns={
    "uoc_luong_tich_duong_dan": "phase2_estimate",
    "CI_95_duoi": "delta_reference_low",
    "CI_95_tren": "delta_reference_high",
})
comparison = comparison.merge(
    delta_rerun[["bien_tang_I", "est", "ci.lower", "ci.upper"]].rename(columns={
        "est": "delta_refit_estimate", "ci.lower": "delta_refit_low", "ci.upper": "delta_refit_high"
    }), on="bien_tang_I", how="inner"
)
comparison = comparison.merge(
    mc_lavaan[["bien_tang_I", "est", "se", "ci.lower", "ci.upper"]].rename(columns={
        "est": "mc_lavaan_estimate", "se": "SE_MC_lavaan", "ci.lower": "mc_lavaan_low", "ci.upper": "mc_lavaan_high"
    }), on="bien_tang_I", how="inner"
)
comparison = comparison.merge(
    mc_manual[["bien_tang_I", "uoc_luong_trung_binh", "trung_vi", "sai_so_chuan_MC", "CI_MC_duoi", "CI_MC_tren", "p_MC_hai_phia", "ty_le_duong"]].rename(columns={
        "uoc_luong_trung_binh": "mc_independent_mean", "trung_vi": "mc_independent_median",
        "sai_so_chuan_MC": "mc_independent_se", "CI_MC_duoi": "mc_independent_low",
        "CI_MC_tren": "mc_independent_high", "p_MC_hai_phia": "mc_independent_p",
        "ty_le_duong": "mc_positive_fraction",
    }), on="bien_tang_I", how="inner"
)
if len(comparison) != 3:
    raise ValueError("Cannot join all three indirect effects.")

def excludes_zero(lower, upper):
    return bool((lower > 0 and upper > 0) or (lower < 0 and upper < 0))

comparison["delta_excludes_zero"] = [excludes_zero(lo, hi) for lo, hi in zip(comparison["delta_refit_low"], comparison["delta_refit_high"])]
comparison["mc_lavaan_excludes_zero"] = [excludes_zero(lo, hi) for lo, hi in zip(comparison["mc_lavaan_low"], comparison["mc_lavaan_high"])]
comparison["mc_independent_excludes_zero"] = [excludes_zero(lo, hi) for lo, hi in zip(comparison["mc_independent_low"], comparison["mc_independent_high"])]
comparison["estimate_matches_phase2"] = (comparison["mc_lavaan_estimate"] - comparison["phase2_estimate"]).abs() <= REFERENCE_TOLERANCE

def classify(row):
    mc_both = bool(row["mc_lavaan_excludes_zero"] and row["mc_independent_excludes_zero"])
    same_positive = bool(row["mc_lavaan_low"] > 0 and row["mc_independent_low"] > 0 and row["phase2_estimate"] > 0)
    same_negative = bool(row["mc_lavaan_high"] < 0 and row["mc_independent_high"] < 0 and row["phase2_estimate"] < 0)
    if mc_both and same_negative:
        return "ROBUST: both Monte Carlo checks support a negative indirect effect"
    if mc_both and same_positive:
        return "ROBUST: both Monte Carlo checks support a positive indirect effect"
    if row["delta_excludes_zero"] and not mc_both:
        return "FRAGILE: Delta excludes zero but at least one Monte Carlo interval includes zero"
    return "INSUFFICIENT EVIDENCE: a nonzero indirect effect was not established"

comparison["robustness_conclusion"] = comparison.apply(classify, axis=1)
comparison.to_csv(output_dir / "11_so_sanh_khoang_tin_cay_va_ket_luan.csv", index=False, encoding="utf-8-sig")
display(comparison)

In [ ]:
method_specs = [
    ("Delta", "delta_refit_estimate", "delta_refit_low", "delta_refit_high", "#4C78A8"),
    ("MC lavaan", "mc_lavaan_estimate", "mc_lavaan_low", "mc_lavaan_high", "#F58518"),
    ("Independent MC", "mc_independent_median", "mc_independent_low", "mc_independent_high", "#54A24B"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=False)
for ax, (_, row) in zip(axes, comparison.sort_values("bien_tang_I").iterrows()):
    for y, (method, center_col, lower_col, upper_col, color) in enumerate(method_specs):
        center = float(row[center_col])
        lower = float(row[lower_col])
        upper = float(row[upper_col])
        ax.errorbar(center, y, xerr=[[center-lower], [upper-center]], fmt="o", color=color, capsize=5, label=method)
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_yticks(range(3), [x[0] for x in method_specs])
    ax.set_title(str(row["bien_tang_I"]), fontweight="bold")
    ax.set_xlabel("Unstandardized indirect effect")
fig.suptitle("Comparison of 95% confidence intervals", fontweight="bold", fontsize=14)
fig.tight_layout()
fig.savefig(output_dir / "12_bieu_do_so_sanh_khoang_tin_cay.png", dpi=180, bbox_inches="tight")
plt.show()

draw_sample = pd.read_csv(output_dir / "07_mau_phan_phoi_MC_de_ve.csv")
long_draws = draw_sample.melt(id_vars="lan_lay_mau", value_vars=["ENP", "PTED", "MEU"], var_name="Tier I variable", value_name="Indirect effect")
grid = sns.displot(long_draws, x="Indirect effect", col="Tier I variable", kind="hist", bins=55, facet_kws={"sharex": False, "sharey": False}, color="#4C78A8", height=4, aspect=1.15)
for ax in grid.axes.flat:
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
grid.figure.suptitle("Monte Carlo simulation distribution", y=1.05, fontweight="bold")
grid.savefig(output_dir / "13_phan_phoi_Monte_Carlo.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
def fmt(value, digits=4):
    return f"{float(value):.{digits}f}"


paragraphs = []
for _, row in comparison.sort_values("bien_tang_I").iterrows():
    variable = str(row["bien_tang_I"])
    paragraphs.append(
        f"### {variable}\n\n"
        f"The full indirect-effect estimate is {fmt(row['phase2_estimate'])}. "
        f"The refitted 95% Delta interval is [{fmt(row['delta_refit_low'])}; {fmt(row['delta_refit_high'])}]. "
        f"The lavaan Monte Carlo interval is [{fmt(row['mc_lavaan_low'])}; {fmt(row['mc_lavaan_high'])}], "
        f"and the independent simulation yields [{fmt(row['mc_independent_low'])}; {fmt(row['mc_independent_high'])}].\n\n"
        f"Conclusion: {row['robustness_conclusion']}."
    )

all_robust = comparison["robustness_conclusion"].str.startswith("ROBUST").all()
pted_row = comparison.loc[comparison["bien_tang_I"] == "PTED"].iloc[0]
if str(pted_row["robustness_conclusion"]).startswith("ROBUST"):
    pted_summary = "Both Monte Carlo intervals for PTED exclude zero, strengthening its result relative to Delta alone."
else:
    pted_summary = "The PTED indirect effect is not stable under Monte Carlo checks; do not claim statistical support for the PTED chain."

overall = (
    "All three chains met the Monte Carlo robustness criteria." if all_robust
    else "The three chains do not all meet robustness criteria; report each tier I variable separately."
)

report_md = f'''# Monte Carlo assessment of indirect-effect robustness

## 1. Objective

Examine whether the full indirect effects from ENP, PTED, and MEU to BI through STRATEGY, FUNCTIONAL_NEEDS, and PU remain supported without assuming that the product of four coefficients has a symmetric normal distribution.

## 2. Method

Keep the WLSMV SEM and its {n_primary} respondents fixed. Evaluate each indirect effect using Delta, lavaan Monte Carlo, and {MC_DRAWS:,} independent joint multivariate Monte Carlo draws.

## 3. Results by tier I variable

{chr(10).join(paragraphs)}

## 4. Overall conclusion

{overall} {pted_summary}

These results test whether observed structural associations and indirect effects are consistent with the proposed theoretical order. A single observational survey cannot establish causal effects.
'''
(output_dir / "14_bao_cao_dien_giai_Monte_Carlo.md").write_text(report_md, encoding="utf-8")

rows_html = []
for _, row in comparison.sort_values("bien_tang_I").iterrows():
    rows_html.append(
        "<tr>" +
        f"<td>{html.escape(str(row['bien_tang_I']))}</td>" +
        f"<td>{fmt(row['phase2_estimate'])}</td>" +
        f"<td>[{fmt(row['delta_refit_low'])}; {fmt(row['delta_refit_high'])}]</td>" +
        f"<td>[{fmt(row['mc_lavaan_low'])}; {fmt(row['mc_lavaan_high'])}]</td>" +
        f"<td>[{fmt(row['mc_independent_low'])}; {fmt(row['mc_independent_high'])}]</td>" +
        f"<td>{html.escape(str(row['robustness_conclusion']))}</td>" +
        "</tr>"
    )

report_html = f'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><title>Monte Carlo report</title>
<style>body{{font-family:Arial,sans-serif;max-width:1100px;margin:32px auto;line-height:1.55;color:#111}}table{{border-collapse:collapse;width:100%}}th,td{{border:1px solid #333;padding:8px;vertical-align:top}}th{{background:#f2f2f2}}img{{max-width:100%}}h1,h2{{color:#111}}</style></head>
<body><h1>Monte Carlo robustness of indirect effects</h1>
<h2>Objective</h2><p>Check three full indirect effects in the prespecified SEM.</p>
<h2>Results</h2><table><thead><tr><th>Tier I variable</th><th>Estimate</th><th>Delta 95%</th><th>MC lavaan 95%</th><th>Independent MC 95%</th><th>Conclusion</th></tr></thead>
<tbody>{''.join(rows_html)}</tbody></table>
<h2>Overall conclusion</h2><p>{html.escape(overall)} {html.escape(pted_summary)}</p>
<p>The cross sectional design does not provide causal evidence.</p>
<img src="12_bieu_do_so_sanh_khoang_tin_cay.png" alt="Confidence interval comparison"></body></html>'''
(output_dir / "15_bao_cao_tong_hop_Monte_Carlo.html").write_text(report_html, encoding="utf-8")

print(overall)
print(pted_summary)

In [ ]:
expected_outputs = {
    "01_kiem_tra_dau_vao.csv": 5,
    "02_fit_xac_nhan.csv": 1,
    "04_tac_dong_gian_tiep_MC_lavaan.csv": 3,
    "05_tac_dong_gian_tiep_Delta_chay_lai.csv": 3,
    "06_tac_dong_gian_tiep_MC_doc_lap.csv": 3,
    "09_doi_chieu_do_phu_hop.csv": 4,
    "11_so_sanh_khoang_tin_cay_va_ket_luan.csv": 3,
}
readback_rows = []
for filename, expected_n in expected_outputs.items():
    path = output_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing output: {filename}")
    frame = pd.read_csv(path, encoding="utf-8-sig")
    ok = len(frame) == expected_n
    readback_rows.append([filename, len(frame), expected_n, ok])
    if not ok:
        raise ValueError(f"{filename} has {len(frame)} rows; expected {expected_n}.")

if not comparison["estimate_matches_phase2"].all():
    raise ValueError("The indirect-effect estimate does not reproduce the phase 2 estimate.")
if comparison["robustness_conclusion"].isna().any():
    raise ValueError("A robustness conclusion is missing.")

readback = pd.DataFrame(readback_rows, columns=["file", "rows_read_back", "expected_rows", "passed"])
readback.to_csv(output_dir / "16_kiem_tra_doc_lai_dau_ra.csv", index=False, encoding="utf-8-sig")

manifest_rows = []
for path in sorted(output_dir.iterdir()):
    if path.is_file() and path.name not in {"17_manifest_sha256.csv"}:
        manifest_rows.append([path.name, sha256_file(path), path.stat().st_size])
manifest_out = pd.DataFrame(manifest_rows, columns=["file", "sha256", "size_bytes"])
manifest_out.to_csv(output_dir / "17_manifest_sha256.csv", index=False, encoding="utf-8-sig")

# Verify the manifest immediately after creation.
manifest_readback = pd.read_csv(output_dir / "17_manifest_sha256.csv", encoding="utf-8-sig")
for _, row in manifest_readback.iterrows():
    path = output_dir / str(row["file"])
    if sha256_file(path) != str(row["sha256"]):
        raise ValueError(f"File changed after creating its manifest: {path.name}")

if sha256_file(input_path) != input_hash_before_r:
    raise RuntimeError('Survey CSV changed during analysis.')
output_dir = main_output_dir
archive_path = shutil.make_archive(str(main_output_dir), 'zip', root_dir=main_output_dir.parent, base_dir=main_output_dir.name)
print('PHASE 2 COMPLETE. Directory:', main_output_dir, '\nZIP:', archive_path)


## Interpretation and limitations

Report each estimated relationship, confidence interval, model diagnostics, and sensitivity result. Cross sectional data cannot establish temporal causality.
